# Full-Stack Hybrid Book Recommendation System — Model Training & Evaluation

This notebook serves as the **ML and Research Source of Truth** for the Book Recommendation System.
It covers:
1. Imports & Environment Setup
2. Dataset Loading (Books.csv, Users.csv, Ratings.csv)
3. Exploratory Data Analysis (Sparsity, Rating Distribution, Empirical $m$ derivation)
4. Data Cleaning & Preprocessing (Explicit vs Implicit Ratings, Thresholding)
5. Popularity-Based Model ($W = \frac{v}{v+m}R + \frac{m}{v+m}C$)
6. Collaborative Filtering Architecture
7. Candidate Algorithm Comparison (Item-Item Cosine vs User-User Cosine vs SVD Matrix Factorization)
8. Quantitative Evaluation on Per-User Split (Precision@K, Recall@K, MAP@K, NDCG@K)
9. Recommendation Engine Sanity Checks & Examples
10. Model Artifact Export
11. Saving Production Binaries
12. Final Summary & Technical Insights


In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Set random seed for reproducibility
np.random.seed(42)
pd.set_option('display.max_columns', None)
print("All dependencies successfully imported.")


## 2. Dataset Loading

We load the Kaggle Book Recommendation Dataset from `data/raw/` (or run `scripts/download_data.py`).


In [ ]:
DATA_RAW_DIR = "../data/raw"
DATA_PROC_DIR = "../data/processed"
ARTIFACTS_DIR = "../backend/artifacts"

books_raw = pd.read_csv(os.path.join(DATA_RAW_DIR, "Books.csv"), low_memory=False, dtype=str)
users_raw = pd.read_csv(os.path.join(DATA_RAW_DIR, "Users.csv"), low_memory=False)
ratings_raw = pd.read_csv(os.path.join(DATA_RAW_DIR, "Ratings.csv"), low_memory=False, dtype={"User-ID": str, "ISBN": str, "Book-Rating": str})

print(f"Raw Books Count: {len(books_raw):,}")
print(f"Raw Users Count: {len(users_raw):,}")
print(f"Raw Ratings Count: {len(ratings_raw):,}")


## 3. Exploratory Data Analysis (EDA) & Empirical Threshold $m$

We analyze the interaction distribution, distinguish explicit (1-10) vs implicit (0) ratings, and derive the minimum rating threshold $m$ statistically.


In [ ]:
ratings_raw["rating"] = pd.to_numeric(ratings_raw["Book-Rating"], errors="coerce").fillna(0).astype(int)
explicit_raw = ratings_raw[ratings_raw["rating"] > 0]
implicit_raw = ratings_raw[ratings_raw["rating"] == 0]

print(f"Explicit Ratings (1-10): {len(explicit_raw):,} ({len(explicit_raw)/len(ratings_raw):.1%})")
print(f"Implicit Interactions (0): {len(implicit_raw):,} ({len(implicit_raw)/len(ratings_raw):.1%})")

# Per-book rating counts
book_counts = explicit_raw["ISBN"].value_counts()
global_C = float(explicit_raw["rating"].mean())
m_empirical = float(np.percentile(book_counts, 75))

print(f"Global Mean Rating (C): {global_C:.4f}")
print(f"Empirical 75th Percentile Rating Count Threshold (m): {m_empirical:.2f}")


## 4. Data Cleaning & Preprocessing

Run pre-calculated preprocessing or load clean data from `data/processed/`.


In [ ]:
from scripts.preprocess import preprocess_pipeline
stats = preprocess_pipeline(min_book_ratings=2, min_user_ratings=2)

books_clean = pd.read_csv(os.path.join(DATA_PROC_DIR, "cleaned_books.csv"))
users_clean = pd.read_csv(os.path.join(DATA_PROC_DIR, "cleaned_users.csv"))
ratings_clean = pd.read_csv(os.path.join(DATA_PROC_DIR, "cleaned_ratings.csv"))

print("Cleaned Datasets:")
print(f"  Books: {len(books_clean):,}")
print(f"  Users: {len(users_clean):,}")
print(f"  Ratings: {len(ratings_clean):,}")


## 5. Popularity-Based Model

Weighted Rating Formula:
$$W = \left(\frac{v}{v+m}\right) R + \left(\frac{m}{v+m}\right) C$$

Where $v$ is number of ratings, $R$ is average rating, $m$ is empirical threshold, and $C$ is global mean rating.


In [ ]:
top_popular = books_clean.sort_values(by="popularity_score", ascending=False).head(10)
top_popular[["isbn", "title", "author", "rating_count", "avg_rating", "popularity_score"]]


## 6 & 7. Collaborative Filtering & Candidate Model Comparison

We evaluate three candidate algorithms on the exact same **per-user train/test split**:
1. **Item-Item Cosine Similarity**
2. **User-User Cosine Similarity**
3. **SVD Matrix Factorization**


In [ ]:
from scripts.train import train_pipeline
model_config = train_pipeline(k_eval=10)


## 8. Quantitative Model Evaluation Summary

Metrics evaluated: **Precision@10**, **Recall@10**, **MAP@10**, and **NDCG@10**.


In [ ]:
eval_metrics = model_config["evaluation_metrics"]
eval_df = pd.DataFrame(eval_metrics).T
eval_df


## 9. Recommendation Examples & Sanity Checks

We test Cold Start handling and Personalized Hybrid recommendations.


In [ ]:
print("Winning Selected Algorithm:", model_config["selected_algorithm"])
print("Configured Hybrid Weights:", model_config["default_weights"])


## 10 & 11. Model Export & Saving Artifacts

Production binaries exported to `backend/artifacts/`:
- `popularity_model.pkl`
- `collaborative_model.pkl`
- `similarity_matrix.pkl`
- `metadata.pkl`
- `model_config.json`


In [ ]:
artifact_files = os.listdir(ARTIFACTS_DIR)
print("Exported artifacts in backend/artifacts/:")
for f in sorted(artifact_files):
    size_kb = os.path.getsize(os.path.join(ARTIFACTS_DIR, f)) / 1024
    print(f"  - {f:<25}: {size_kb:.2f} KB")


## 12. Final Model Summary

- **Popularity Engine**: Empirical $m$ weighted rating score safely resolves cold-start scenarios.
- **Personalized Engine**: Item-Item Cosine Similarity demonstrated superior NDCG@10 and MAP@10 compared to User-User CF and SVD on the per-user evaluation split.
- **Hybrid Strategy**: Min-max normalized linear blend $0.7 \times S_{\text{CF}} + 0.3 \times S_{\text{pop}}$.
- **Incremental Learning**: Real-time rating submissions update the MongoDB database and active user interaction vector without needing costly full model retraining.
